# OSL SAC Notebook — Bilateral Sensor + Larva Connectome (off-policy)

Same pipeline as the PPO notebook (parallel envs, curriculum, connectome actor, logging, eval) — only the optimisation algorithm differs.

Setup
- Env: `OslEnv` — bilateral sensors (0.15 mm), independent head/body rotation, obs `[c_L, c_R, prev_v, prev_body_omega, prev_head_omega]`, action `[v, body_omega, head_omega]`.
- Actor: `Connectome` (387 base nodes + 2 sensor input + 32 latent output, ~17k learnable edges, 6 message-passing steps) → concat efference copy → tanh-squashed Gaussian.
- Critics: twin Q networks Q(s, a) → scalar, with Polyak-averaged targets.
- Temperature α: auto-tuned to track target entropy = −action_dim.
- Trainer: off-policy SAC, replay buffer, parallel rollout + minibatch updates.

Repo clone is mandatory — connectome CSVs live in `assets/connectome/`.

In [ ]:
# Colab setup. Re-running is safe. Skip this cell when running locally.
import os, sys, subprocess

REPO_URL = 'https://github.com/InHyunseo/Brain-inspired-OSL.git'
REPO_DIR = '/content/2d-osl' if os.path.isdir('/content') else os.path.abspath('2d-osl')
if not os.path.isdir(REPO_DIR):
    subprocess.check_call(['git', 'clone', REPO_URL, REPO_DIR])
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Colab already has torch / numpy / gymnasium / matplotlib / tensorboard / imageio
# pre-installed — skip pip install to save ~30s of resolver work.

for p in ('assets/connectome/weights.csv', 'assets/connectome/metadata.csv'):
    assert os.path.exists(p), f'Missing {p} — clone failed?'
print('repo:', REPO_DIR, '\ncwd :', os.getcwd())


In [ ]:
# ===== Local setup (skip the Colab cell above; run this instead) =====
# Assumes you already have the repo cloned locally and dependencies installed.
# Edit REPO_DIR if your checkout is elsewhere.
import os, sys

REPO_DIR = os.path.abspath('..')  # notebook lives at <repo>/ipynb/, so .. is the repo root
if not os.path.isdir(os.path.join(REPO_DIR, 'assets', 'connectome')):
    REPO_DIR = os.path.abspath('.')  # fallback: notebook opened with repo as cwd

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

for p in ('assets/connectome/weights.csv', 'assets/connectome/metadata.csv'):
    assert os.path.exists(p), f'Missing {p} — wrong REPO_DIR? cwd={os.getcwd()}'

print('repo:', REPO_DIR)
print('cwd :', os.getcwd())


## Smoke check

In [ ]:
import torch, numpy as np
from src.envs.osl_env import OslEnv
from src.agents.sac_agent import SACPolicy

env = OslEnv()
obs, info = env.reset(seed=0)
print('obs', obs.shape, 'action_space', env.action_space.shape)

policy = SACPolicy(
    weights_csv='assets/connectome/weights.csv',
    metadata_csv='assets/connectome/metadata.csv',
    latent_dim=32, message_passing_steps=6,
)
print('backbone:', policy.backbone.describe())
print('actor params :', sum(p.numel() for p in policy.actor_parameters()))
print('critic params:', sum(p.numel() for p in policy.critic_parameters()))

## Training

All knobs are plain Python variables — change them, re-run the cell. The full plan default is `[(0,0.0,1500000),(1,0.3,500000),(1,0.6,500000),(2,1.0,1000000)]` (3.5M total). The notebook ships a smaller Colab-friendly schedule so you can iterate; bump the per-phase counts when ready for a real run.

### Live TensorBoard

In [ ]:
# ===== TensorBoard (run BEFORE the training cell to watch curves live) =====
# Works in Colab and local Jupyter. Writers go to <RUN_DIR>/tb/ inside the run.
import os
os.makedirs('runs', exist_ok=True)
%load_ext tensorboard
%tensorboard --logdir runs --port 6006


In [ ]:
# ===== HYPERPARAMETERS — edit freely =====
# Curriculum: list of (noise_stage, noise_strength, env_steps).
# Bump-field noise model (see src/envs/odor_field.py) — identical to PPO notebook.
PHASES = [
    # --- Stage 0 (clean): success-radius curriculum, spawn fixed at 55-70mm ---
    # Start with a loose goal radius (20mm) so the agent gets early successes,
    # then tighten toward the real 7.5mm target. 1M steps per radius.
    # connectome backbone learns slower than GRU (late start from the cast/idle
    # structural default, 6-hop recurrent + sparse gradient) so the clean stage
    # gets a larger budget here than the GRU notebooks (final-performance comparison).
    (0, 0.0, 2_000_000, 20.0),
    (0, 0.0, 2_000_000, 15.0),
    (0, 0.0, 2_000_000, 10.0),
    (0, 0.0, 3_000_000,  7.5),   # real target; extra steps to consolidate
    # --- Noise curriculum (goal radius fixed at 7.5mm); stop at 0.3 (beyond it fails) ---
    (1, 0.3, 1_000_000,  7.5),
    (2, 0.3, 1_000_000,  7.5),
]

# Env
ENV_KW = dict(
    sensor_spacing_mm=0.15,
    episode_seconds=120.0,
    arena_width_mm=80.0, arena_height_mm=120.0,
    source_x_mm=40.0, source_y_mm=100.0,
    gaussian_sigma_mm=30.0, success_radius_mm=7.5,
)

# Trainer (SACConfig). Off-policy: rollout_steps × num_envs transitions are
# collected, then gradient_steps minibatch updates run from the replay buffer.
SAC_KW = dict(
    rollout_steps=32, num_envs=8, parallel_envs=True,
    gradient_steps=32, batch_size=256,
    buffer_capacity=200_000,
    learning_starts_steps=5_000,
    gamma=0.99, tau=0.005,
    actor_lr=3e-4, critic_lr=3e-4, alpha_lr=3e-4,
    target_entropy=-0.5,  # was None(-> -3 auto): -3 collapsed alpha to ~0 in 40k steps,
                          # killing exploration before any success. -1.0 keeps casting/exploration alive.
    actor_max_grad_norm=0.5, critic_max_grad_norm=0.5,
    log_std_init=-0.5,
    latent_dim=32, message_passing_steps=6,
    weights_csv='assets/connectome/weights.csv',
    metadata_csv='assets/connectome/metadata.csv',
    critic_hidden=(128, 128),
    eval_interval_updates=10, eval_episodes=3,
    log_every_updates=1, checkpoint_every_timesteps=100_000,
    seed=0,
    device='auto',
)
# ==========================================

import os, time, json, torch
from src.agents.sac_agent import SACConfig, SACTrainer

RUN_DIR = os.path.join('runs', f'sac_nb_{time.strftime("%Y%m%d_%H%M%S")}')
os.makedirs(os.path.join(RUN_DIR, 'plots'), exist_ok=True)
with open(os.path.join(RUN_DIR, 'config.json'), 'w') as f:
    json.dump({'env': ENV_KW, 'sac': SAC_KW, 'phases': PHASES}, f, indent=2)
print('[run_dir]', RUN_DIR)

env_cfg = {**ENV_KW, 'noise_stage': 0, 'noise_strength': 0.0, 'seed': SAC_KW['seed']}
cfg = SACConfig(**SAC_KW)

trainer = SACTrainer(env_cfg, cfg, run_dir=RUN_DIR)
summary = {}
try:
    for stage, strength, steps, goal_radius in PHASES:
        print(f'\n=== Phase noise_stage={stage} strength={strength} goal_radius={goal_radius} steps={steps} ===')
        trainer.runner.set_noise_stage(stage, strength)
        trainer.runner.set_success_radius(goal_radius)
        trainer.env_config['success_radius_mm'] = goal_radius  # so eval uses the same goal
        summary = trainer.train(phase_timesteps=steps)
    trainer.save_final(summary)
    print('\n[final summary]\n' + json.dumps(summary, indent=2))
finally:
    trainer.close()


## Curriculum 냄새 농도 분포 시각화
각 커리큘럼 phase의 `(noise_stage, noise_strength=α)`에 대해 bump-field 농도 분포를 그립니다.
- **Stage 0**: clean Gaussian.
- **Stage 1**: static bump field — bump들이 reset 시점에 고정.
- **Stage 2**: dynamic bump field — bump들이 drift + AR(1) 진폭 + 확률적 respawn 으로 시공간적으로 변함.

Stage 2 phase는 inline 애니메이션(HTML5)으로, 그 외는 정적 heatmap. cyan 실선/점선은 spawn 영역(`c ≥ c_thresh`)과 min-radius 제외 원, 라임 점/화살표는 실제로 `env.reset()`이 만들어내는 spawn 위치 + 헤딩.


In [ ]:
# ===== Curriculum odor-field viewer =====
# Requires PHASES and ENV_KW from the hyperparameter cell above.
import math
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.patches import Circle
from IPython.display import HTML, display

from src.envs.osl_env import EnvConfig, OslEnv

VIZ_GRID_MM = 0.5         # heatmap resolution (mm/pixel) — coarser than the noise grid for speed
ANIM_FRAMES = 60          # frames for Stage 2 animation
ANIM_INTERVAL_MS = 80
SUCCESS_RADIUS_MM = ENV_KW.get('success_radius_mm', 7.5)
SPAWN_SAMPLES = 40        # number of spawn (x, y, heading) samples to overlay per phase
HEADING_ARROW_MM = 6.0    # arrow length for spawn heading visualization

def _make_env(noise_stage, noise_strength, seed=0):
    cfg_dict = {**ENV_KW, 'noise_stage': int(noise_stage),
                'noise_strength': float(noise_strength), 'seed': seed}
    return OslEnv(EnvConfig.from_dict(cfg_dict))

def _sample_field(env):
    W = env.cfg.arena_width_mm
    H = env.cfg.arena_height_mm
    xs = np.arange(0.0, W + VIZ_GRID_MM, VIZ_GRID_MM)
    ys = np.arange(0.0, H + VIZ_GRID_MM, VIZ_GRID_MM)
    grid = np.empty((len(ys), len(xs)), dtype=np.float32)
    for j, y in enumerate(ys):
        for i, x in enumerate(xs):
            grid[j, i] = env.field.sample(float(x), float(y))
    return xs, ys, grid

def _base_grid(env):
    W = env.cfg.arena_width_mm
    H = env.cfg.arena_height_mm
    xs = np.arange(0.0, W + VIZ_GRID_MM, VIZ_GRID_MM)
    ys = np.arange(0.0, H + VIZ_GRID_MM, VIZ_GRID_MM)
    X, Y = np.meshgrid(xs, ys)
    dx = X - env.cfg.source_x_mm
    dy = Y - env.cfg.source_y_mm
    g = env.cfg.c_background + env.cfg.c_peak * np.exp(
        -(dx * dx + dy * dy) / (2.0 * env.cfg.gaussian_sigma_mm ** 2))
    return xs, ys, g

def _draw_spawn_overlay(ax, env, n_samples=SPAWN_SAMPLES):
    # Threshold contour for the spawnable region (base field).
    xs, ys, gbase = _base_grid(env)
    c_thr = env.cfg.spawn_c_thresh_frac * env.cfg.c_peak
    ax.contour(xs, ys, gbase, levels=[c_thr],
               colors='cyan', linewidths=1.2, linestyles='-')
    # Min-radius exclusion around source.
    ax.add_patch(Circle((env.cfg.source_x_mm, env.cfg.source_y_mm),
                        env.cfg.spawn_min_radius_mm, fill=False,
                        edgecolor='cyan', linestyle=':', linewidth=1.0))
    # Sample spawns via the env's own reset() so we visualize the real policy.
    sx, sy, sh = [], [], []
    for k in range(n_samples):
        env.reset(seed=10_000 + k)
        sx.append(env.x_mm); sy.append(env.y_mm); sh.append(env.heading_rad)
    sx = np.asarray(sx); sy = np.asarray(sy); sh = np.asarray(sh)
    ax.scatter(sx, sy, s=14, c='lime', edgecolors='black',
               linewidths=0.4, zorder=5, label='spawn')
    ax.quiver(sx, sy, HEADING_ARROW_MM * np.cos(sh), HEADING_ARROW_MM * np.sin(sh),
              angles='xy', scale_units='xy', scale=1.0,
              color='lime', width=0.004, zorder=5)

def _decorate(ax, env, title, with_spawn=True):
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('x (mm)'); ax.set_ylabel('y (mm)')
    ax.set_aspect('equal')
    sx, sy = env.cfg.source_x_mm, env.cfg.source_y_mm
    ax.plot([sx], [sy], marker='*', color='white', markersize=14,
            markeredgecolor='black', linewidth=0)
    ax.add_patch(Circle((sx, sy), SUCCESS_RADIUS_MM, fill=False,
                        edgecolor='white', linestyle='--', linewidth=1.0))
    if with_spawn:
        _draw_spawn_overlay(ax, env)

# ---- 1) Static snapshot per phase ----
n = len(PHASES)
fig, axes = plt.subplots(1, n, figsize=(4.2 * n, 5.2), squeeze=False)
for k, (stage, strength, steps, goal_radius) in enumerate(PHASES):
    env = _make_env(stage, strength, seed=42 + k)
    xs, ys, grid = _sample_field(env)
    ax = axes[0, k]
    im = ax.imshow(grid, origin='lower',
                   extent=[xs[0], xs[-1], ys[0], ys[-1]],
                   cmap='magma', vmin=0.0)
    _decorate(ax, env,
              f'Phase {k}: stage={stage} strength={strength}\nsteps={steps:,}')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='c')
fig.suptitle('Curriculum odor-concentration snapshots (cyan = spawn region, '
             'lime = sampled spawns + heading)', y=1.02, fontsize=11)
fig.tight_layout()
plt.show()

# ---- 2) Temporal animation for any stage-2 phase ----
stage2_phases = [(i, p) for i, p in enumerate(PHASES) if p[0] >= 2]
if not stage2_phases:
    print('No stage-2 phase in PHASES — skipping temporal animation.')
else:
    for k, (stage, strength, steps, goal_radius) in stage2_phases:
        env = _make_env(stage, strength, seed=100 + k)
        xs, ys, grid0 = _sample_field(env)
        fig, ax = plt.subplots(figsize=(5.0, 6.2))
        im = ax.imshow(grid0, origin='lower',
                       extent=[xs[0], xs[-1], ys[0], ys[-1]],
                       cmap='magma', vmin=0.0,
                       vmax=max(1e-6, float(grid0.max()) * 1.2))
        _decorate(ax, env,
                  f'Phase {k} (animated): stage={stage} strength={strength}',
                  with_spawn=False)  # animate background field only; no static spawn dots
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='c')

        def _update(frame_idx, env=env, im=im):
            env.field.advance()
            _, _, g = _sample_field(env)
            im.set_data(g)
            return (im,)

        anim = animation.FuncAnimation(fig, _update, frames=ANIM_FRAMES,
                                       interval=ANIM_INTERVAL_MS, blit=True)
        plt.close(fig)
        display(HTML(anim.to_jshtml()))


## Training curves

In [ ]:
import json, os
import matplotlib.pyplot as plt

log_path = os.path.join(RUN_DIR, 'training_log.jsonl')
rows = [json.loads(l) for l in open(log_path)]
steps = [r['total_steps'] for r in rows]

fig, ax = plt.subplots(2, 3, figsize=(15, 7))
ax[0,0].plot(steps, [r['recent_return_mean'] for r in rows]);   ax[0,0].set_title('recent return'); ax[0,0].grid(alpha=.3)
ax[0,1].plot(steps, [r['recent_success_rate'] for r in rows]);  ax[0,1].set_title('success rate'); ax[0,1].grid(alpha=.3)
ax[0,2].plot(steps, [r.get('recent_cast_mean', 0.0) for r in rows]); ax[0,2].set_title('mean casts / episode'); ax[0,2].grid(alpha=.3)
ax[1,0].plot(steps, [r['actor_loss'] for r in rows], label='actor')
ax[1,0].plot(steps, [r['critic_loss'] for r in rows], label='critic'); ax[1,0].legend(); ax[1,0].set_title('losses'); ax[1,0].grid(alpha=.3)
ax[1,1].plot(steps, [r.get('alpha', 0.0) for r in rows]);        ax[1,1].set_title('SAC α (entropy temp)'); ax[1,1].grid(alpha=.3)
ax[1,2].plot(steps, [r.get('q_mean', 0.0) for r in rows]);       ax[1,2].set_title('mean Q (new actions)'); ax[1,2].grid(alpha=.3)
for a in ax.flat: a.set_xlabel('env steps')
fig.tight_layout(); plt.show()


## Evaluation: elite-seed list + render
**셀 A** (아래 첫 코드 셀): policy 로드 + elite 시드 목록 만들기. `EVAL_*` / `ELITE_*` 노브 바꾸려면 이 셀만 재실행.

**셀 B** (그 다음): `PICK_INDEX = 3` 처럼 인덱스만 바꾸고 셀 B만 재실행하면 GIF 다시 그림. `EXPLICIT_SEED = 20137`로 elite 목록 무시하고 임의 시드도 가능.

> Determinism note — `env.reset(seed=...)`가 bump-field rng까지 같은 시드로 재시드해서, 같은 시드를 다시 그리면 trajectory + plume 시각화 둘 다 완전히 동일하게 재현됩니다.

In [ ]:
# ===== Cell A: Load policy + find elite seeds =====
# Rerun whenever you want to refresh the elite list (e.g. change noise stage,
# strength, cast filter, or trial budget).
EVAL_NOISE_STAGE = 2
EVAL_NOISE_STRENGTH = 1.0      # bump-field α at eval time
EVAL_SEED_BASE = 20_000
EVAL_MAX_TRIALS = 500
N_ELITES = 10                  # collect up to this many elite seeds
ELITE_MIN_CASTS = 1
ELITE_MAX_CASTS = 300
# ====================================================

import os, torch, numpy as np
from src.envs.osl_env import EnvConfig, OslEnv
from src.agents.sac_agent import SACConfig, SACPolicy
from src.utils.plotter import render_rollout_frame, save_gif

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ckpt = torch.load(os.path.join(RUN_DIR, 'ckpt_final.pt'), map_location=device, weights_only=False)
cfg = SACConfig(**ckpt['agent_config'])
policy = SACPolicy(
    weights_csv=cfg.weights_csv, metadata_csv=cfg.metadata_csv,
    latent_dim=cfg.latent_dim, message_passing_steps=cfg.message_passing_steps,
    critic_hidden=cfg.critic_hidden,
    log_std_init=cfg.log_std_init,
    log_std_min=cfg.log_std_min, log_std_max=cfg.log_std_max,
).to(device)
policy.load_state_dict(ckpt['policy_state_dict']); policy.eval()

def make_eval_env(seed):
    cfg_dict = {**ENV_KW, 'noise_stage': EVAL_NOISE_STAGE,
                'noise_strength': EVAL_NOISE_STRENGTH, 'seed': seed}
    return OslEnv(EnvConfig.from_dict(cfg_dict))

def run_episode(seed, collect_traj=False):
    """Deterministic rollout. With same `seed`, trajectory + bump field are
    fully reproducible (env.reset also reseeds field.rng)."""
    env = make_eval_env(seed)
    obs, _ = env.reset(seed=seed)
    actor_state, critic_state = policy.initial_states(1, device)
    mask = torch.zeros(1, 1, device=device)
    ret, casts, success = 0.0, 0, False
    traj_x, traj_y, cast_x, cast_y, frames = [], [], [], [], []
    for t in range(env.max_steps):
        obs_t = torch.as_tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
        with torch.no_grad():
            a, _, n_as, n_cs = policy.act(obs_t, actor_state, critic_state, mask, deterministic=True)
        if collect_traj:
            traj_x.append(env.x_mm); traj_y.append(env.y_mm)
        obs, r, term, trunc, info = env.step(a.squeeze(0).cpu().numpy())
        ret += float(r)
        if info.get('event_is_high_cast_like'):
            casts += 1
            if collect_traj:
                cast_x.append(env.x_mm); cast_y.append(env.y_mm)
        if collect_traj:
            frames.append(render_rollout_frame(env, traj_x, traj_y, cast_x, cast_y, t,
                                               title=f'SAC seed={seed} step={t} casts={casts}'))
        done = bool(term or trunc)
        if done:
            success = bool(info.get('success', False))
            mask.fill_(0.0)
        else:
            mask.fill_(1.0)
        actor_state = n_as * mask; critic_state = n_cs * mask
        if done:
            break
    return {
        'seed': seed, 'return': ret, 'success': success, 'casts': casts,
        'frames': frames if collect_traj else None,
    }

print(f'🔍 elite seed 탐색 (success + casts ∈ [{ELITE_MIN_CASTS}, {ELITE_MAX_CASTS}], '
      f'α={EVAL_NOISE_STRENGTH}, up to {EVAL_MAX_TRIALS} trials)…')
elites, trial = [], 0
while len(elites) < N_ELITES and trial < EVAL_MAX_TRIALS:
    seed = EVAL_SEED_BASE + trial
    r = run_episode(seed, collect_traj=False)
    if r['success'] and ELITE_MIN_CASTS <= r['casts'] <= ELITE_MAX_CASTS:
        elites.append(r)
        print(f"  ✨ [{len(elites)-1:2d}] seed={seed}  ret={r['return']:7.2f}  casts={r['casts']}")
    trial += 1
print(f'\n→ {len(elites)} elite seed(s) after {trial} trials')

if not elites:
    print('⚠️  No elite seed met the criteria — relax ELITE_MIN_CASTS/MAX_CASTS '
          'or raise EVAL_MAX_TRIALS, then rerun this cell.')
else:
    print('\nElite list (index | seed | return | casts):')
    for i, e in enumerate(elites):
        print(f'  [{i:2d}]  seed={e["seed"]}  ret={e["return"]:7.2f}  casts={e["casts"]}')


In [ ]:
# ===== Cell B: Render GIF for a chosen seed =====
# Edit PICK_INDEX (into `elites`) or set EXPLICIT_SEED to any integer.
# Same seed → same trajectory + same plume animation (deterministic).
PICK_INDEX = 0          # index into the elite list printed by Cell A
EXPLICIT_SEED = None    # if set (e.g. 20137), overrides PICK_INDEX
# =================================================

from IPython.display import Image as DisplayImage, display

if EXPLICIT_SEED is not None:
    chosen_seed = int(EXPLICIT_SEED)
    print(f'using EXPLICIT_SEED={chosen_seed}')
else:
    if not elites:
        raise RuntimeError('No elites available — rerun Cell A or set EXPLICIT_SEED.')
    pick = max(0, min(PICK_INDEX, len(elites) - 1))
    chosen_seed = elites[pick]['seed']
    print(f'rendering elites[{pick}] = seed {chosen_seed}')

result = run_episode(chosen_seed, collect_traj=True)
print(f"seed={result['seed']} return={result['return']:.2f} "
      f"success={result['success']} casts={result['casts']}")

gif_path = os.path.join(RUN_DIR, 'plots', f'agent_seed{chosen_seed}.gif')
save_gif(result['frames'], gif_path, fps=15)
display(DisplayImage(data=open(gif_path, 'rb').read(), format='gif'))


## Stochastic evaluation (deterministic=False)
Deterministic 모드는 tanh-Gaussian의 평균만 쓰니까 actor가 entropy를 전혀 안 써 cast가 거의 안 나올 수 있어요. 이 두 셀은 `policy.act(..., deterministic=False)`로 reparam sample 사용 — 진짜 학습된 stochastic policy의 행동을 봅니다. **재현성을 위해 torch global RNG를 매 episode마다 seed에 묶어두니** 같은 seed에 같은 GIF가 나옵니다.

In [ ]:
# ===== Cell C: Stochastic elite seeds =====
# Uses policy.act(deterministic=False) — actor samples from the tanh-Gaussian,
# so behaviour matches what SAC was actually trained against (incl. exploration
# noise from α). Cast events are typically more common here.
# Re-seeding torch.manual_seed(seed) inside run_episode_stoch makes this fully
# reproducible per seed.
STO_EVAL_NOISE_STAGE = 2
STO_EVAL_NOISE_STRENGTH = 1.0
STO_EVAL_SEED_BASE = 30_000     # different base so it doesn't collide with deterministic elites
STO_EVAL_MAX_TRIALS = 500
STO_N_ELITES = 10
STO_ELITE_MIN_CASTS = 1
STO_ELITE_MAX_CASTS = 300
# ==========================================

import os, torch
from src.envs.osl_env import EnvConfig, OslEnv

def make_eval_env_stoch(seed):
    cfg_dict = {**ENV_KW, 'noise_stage': STO_EVAL_NOISE_STAGE,
                'noise_strength': STO_EVAL_NOISE_STRENGTH, 'seed': seed}
    return OslEnv(EnvConfig.from_dict(cfg_dict))

def run_episode_stoch(seed, collect_traj=False):
    """Stochastic deterministic-per-seed rollout. torch.manual_seed(seed) is
    set so the actor's reparam samples are reproducible — same seed → same
    trajectory + same plume animation."""
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    env = make_eval_env_stoch(seed)
    obs, _ = env.reset(seed=seed)
    actor_state, critic_state = policy.initial_states(1, device)
    mask = torch.zeros(1, 1, device=device)
    ret, casts, success = 0.0, 0, False
    traj_x, traj_y, cast_x, cast_y, frames = [], [], [], [], []
    for t in range(env.max_steps):
        obs_t = torch.as_tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
        with torch.no_grad():
            a, _, n_as, n_cs = policy.act(obs_t, actor_state, critic_state, mask, deterministic=False)
        if collect_traj:
            traj_x.append(env.x_mm); traj_y.append(env.y_mm)
        obs, r, term, trunc, info = env.step(a.squeeze(0).cpu().numpy())
        ret += float(r)
        if info.get('event_is_high_cast_like'):
            casts += 1
            if collect_traj:
                cast_x.append(env.x_mm); cast_y.append(env.y_mm)
        if collect_traj:
            frames.append(render_rollout_frame(env, traj_x, traj_y, cast_x, cast_y, t,
                                               title=f'SAC stoch seed={seed} step={t} casts={casts}'))
        done = bool(term or trunc)
        if done:
            success = bool(info.get('success', False))
            mask.fill_(0.0)
        else:
            mask.fill_(1.0)
        actor_state = n_as * mask; critic_state = n_cs * mask
        if done:
            break
    return {
        'seed': seed, 'return': ret, 'success': success, 'casts': casts,
        'frames': frames if collect_traj else None,
    }

print(f'🔍 stochastic elite seed 탐색 (success + casts ∈ '
      f'[{STO_ELITE_MIN_CASTS}, {STO_ELITE_MAX_CASTS}], '
      f'α={STO_EVAL_NOISE_STRENGTH}, up to {STO_EVAL_MAX_TRIALS} trials)…')
elites_stoch, trial = [], 0
while len(elites_stoch) < STO_N_ELITES and trial < STO_EVAL_MAX_TRIALS:
    seed = STO_EVAL_SEED_BASE + trial
    r = run_episode_stoch(seed, collect_traj=False)
    if r['success'] and STO_ELITE_MIN_CASTS <= r['casts'] <= STO_ELITE_MAX_CASTS:
        elites_stoch.append(r)
        print(f"  ✨ [{len(elites_stoch)-1:2d}] seed={seed}  ret={r['return']:7.2f}  casts={r['casts']}")
    trial += 1
print(f'\n→ {len(elites_stoch)} elite seed(s) after {trial} trials')

if not elites_stoch:
    print('⚠️  No stochastic elite seed met the criteria — relax filters '
          'or raise STO_EVAL_MAX_TRIALS, then rerun this cell.')
else:
    print('\nStochastic elite list (index | seed | return | casts):')
    for i, e in enumerate(elites_stoch):
        print(f'  [{i:2d}]  seed={e["seed"]}  ret={e["return"]:7.2f}  casts={e["casts"]}')


In [ ]:
# ===== Cell D: Render stochastic GIF =====
STO_PICK_INDEX = 0          # index into elites_stoch
STO_EXPLICIT_SEED = None    # set to an int to override
# =========================================

from IPython.display import Image as DisplayImage, display

if STO_EXPLICIT_SEED is not None:
    chosen_seed = int(STO_EXPLICIT_SEED)
    print(f'using STO_EXPLICIT_SEED={chosen_seed}')
else:
    if not elites_stoch:
        raise RuntimeError('No stochastic elites — rerun Cell C or set STO_EXPLICIT_SEED.')
    pick = max(0, min(STO_PICK_INDEX, len(elites_stoch) - 1))
    chosen_seed = elites_stoch[pick]['seed']
    print(f'rendering elites_stoch[{pick}] = seed {chosen_seed}')

result = run_episode_stoch(chosen_seed, collect_traj=True)
print(f"seed={result['seed']} return={result['return']:.2f} "
      f"success={result['success']} casts={result['casts']}")

gif_path = os.path.join(RUN_DIR, 'plots', f'agent_stoch_seed{chosen_seed}.gif')
save_gif(result['frames'], gif_path, fps=15)
display(DisplayImage(data=open(gif_path, 'rb').read(), format='gif'))


## Mechanistic analysis (connectome / GRU interpretability)

학습된 정책의 **신경회로 수준 기전**을 분석합니다 (`Analysis.osl2d` 파이프라인 — 3D `osl_analysis`에서 이식). `RUN_DIR/ckpt_final.pt`를 deterministic 롤아웃해 hidden-state trace를 덤프한 뒤 4-phase 분석을 실행합니다:

- **Phase 1** 행동 라벨(run/cast/turn/spin/stop) 분포 + 전이행렬 + cast PSD
- **Phase 2a** hidden state PCA→UMAP 분리도 (silhouette / Calinski-Harabasz)
- **Phase 2b** episode-split linear probe (행동이 hidden에 선형 디코딩되는가)
- **Phase 2c** per-neuron 기여도 → 라벨별 top-K 뉴런 (`neuron_groups.json`)
- **Phase 3a** Jacobian eigenmode (slow attractor / oscillatory mode)
- **Phase 3b** fixed-point 탐색 + 안정성
- **Phase 4** neuron-group ablation 인과 검증 (라이브 env)

> Colab에서 UMAP 시각화를 쓰려면 `pip install umap-learn` (없으면 자동으로 PCA-2 fallback).

In [ ]:
# ===== Analysis Cell 1: dump traces + run the full 4-phase pipeline =====
# Re-uses RUN_DIR from training. Analyzes ckpt_final.pt (label "final").
# Tune ANALYSIS_* knobs for speed vs. coverage.
ANALYSIS_SEEDS = [0, 1]
ANALYSIS_EPISODES_PER_SEED = 8
ANALYSIS_MAX_STEPS = None          # None = full episodes
ANALYSIS_NOISE_STAGE = 2
ANALYSIS_NOISE_STRENGTH = 1.0
ANALYSIS_TOP_K = 16                # top-K neurons per label for Phase 2c/4
ANALYSIS_SAMPLES_PER_LABEL = 60    # Jacobian samples per label (Phase 3a)
RUN_ABLATION = True                # Phase 4 (live env) — set False to skip
ABLATION_EPS = 3
# =======================================================================

import os
from Analysis.osl2d import run_all as _run_all

_argv = [
    '--run-dir', RUN_DIR,
    '--checkpoints', 'final',
    '--seeds', *[str(s) for s in ANALYSIS_SEEDS],
    '--episodes-per-seed', str(ANALYSIS_EPISODES_PER_SEED),
    '--noise-stage', str(ANALYSIS_NOISE_STAGE),
    '--noise-strength', str(ANALYSIS_NOISE_STRENGTH),
    '--top-k', str(ANALYSIS_TOP_K),
    '--samples-per-label', str(ANALYSIS_SAMPLES_PER_LABEL),
    '--ablation-eps', str(ABLATION_EPS),
]
if ANALYSIS_MAX_STEPS is not None:
    _argv += ['--max-steps', str(ANALYSIS_MAX_STEPS)]
if not RUN_ABLATION:
    _argv += ['--skip-ablation']

_run_all.main(_argv)
print('\n[analysis] artifacts →', os.path.join(RUN_DIR, 'analysis'))

In [ ]:
# ===== Analysis Cell 2: display all phase figures inline + key metrics =====
import os, json, glob
from IPython.display import Image as DisplayImage, display, Markdown

ANALYSIS_DIR = os.path.join(RUN_DIR, 'analysis')

# Key scalar metrics
def _safe(path):
    p = os.path.join(ANALYSIS_DIR, path)
    return json.load(open(p)) if os.path.exists(p) else {}

p1 = _safe('phase1_label_stats.json')
p2a = _safe('phase2a_separation.json')
p2b = _safe('phase2b_probe.json').get('overall', {})
p4 = _safe('phase4_ablation.json')

lines = ['### Analysis summary', '']
if p1:
    ratios = ', '.join(f'{k}={v:.2f}' for k, v in p1.get('overall_ratios', {}).items())
    lines.append(f'- **Phase 1** label ratios: {ratios}')
    lines.append(f'  - success_rate={p1.get("success_rate"):.2f}, cast_psd_peak_hz={p1.get("cast_psd_peak_hz")}')
if p2a:
    lines.append(f'- **Phase 2a** silhouette={p2a.get("silhouette")}, calinski_harabasz={p2a.get("calinski_harabasz")}')
if p2b:
    lines.append(f'- **Phase 2b** probe acc={p2b.get("acc")}, shuffle={p2b.get("acc_shuffle")}, macro_f1={p2b.get("macro_f1")}')
if p4 and 'delta' in p4:
    lines.append('- **Phase 4** ablation Δreturn / Δsuccess by group:')
    for g, d in p4['delta'].items():
        lines.append(f'    - `{g}`: Δreturn={d["d_return_mean"]:+.3f}, '
                     f'Δsuccess={d["d_success_rate"]:+.3f}, KL(labels)={d["kl_label_dist_vs_baseline"]:.3f}')
display(Markdown('\n'.join(lines)))

# All figures
for png in sorted(glob.glob(os.path.join(ANALYSIS_DIR, '*.png'))):
    display(Markdown(f'**{os.path.basename(png)}**'))
    display(DisplayImage(data=open(png, 'rb').read(), format='png'))